In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
import re
from stki_utils import (load_corpus, corpus_texts, preprocess, TfidfModel, BM25Model,
                        load_queries, load_qrels, evaluate)


class MesinPencari:
    def __init__(self, model="bm25", use_stopword=True, use_stemming=True):
        self.korpus = load_corpus().set_index("doc_id")
        self.opsi = dict(use_stopword=use_stopword, use_stemming=use_stemming)
        docs = {d: preprocess(t, **self.opsi) for d, t in corpus_texts().items()}
        self.model = BM25Model(docs) if model == "bm25" else TfidfModel(docs)

    def cari(self, query, k=10):
        return [d for d, _ in self.model.search(preprocess(query, **self.opsi), k=k)]

    def tampilkan(self, query, k=3):
        q_terms = set(preprocess(query, **self.opsi))
        print(f"Hasil pencarian untuk: {query!r}\n")
        for rank, (d, s) in enumerate(self.model.search(list(preprocess(query, **self.opsi)), k=k), 1):
            row = self.korpus.loc[d]
            kalimat = re.split(r"(?<=\.)\s+", row.isi)
            # cuplikan: kalimat dengan term query terbanyak
            cuplikan = max(kalimat, key=lambda s_: len(q_terms & set(preprocess(s_, **self.opsi))))
            print(f"{rank}. [{d}] {row.judul}  (skor {s:.3f})\n   ...{cuplikan}\n")
            
queries, qrels = load_queries(), load_qrels()
baris = []
for model in ["tfidf", "bm25"]:
    for sw in [False, True]:
        for st in [False, True]:
            m = MesinPencari(model, use_stopword=sw, use_stemming=st)
            run = {q: m.cari(t) for q, t in queries.items()}
            rata = evaluate(run, qrels, k=5).loc["Rata-rata"]
            baris.append({"model": model, "stopword": sw, "stemming": st,
                          "P@5": rata["P@5"], "MAP": rata["AP"], "nDCG@5": rata["nDCG@5"]})
pd.DataFrame(baris)

,model,stopword,stemming,P@5,MAP,nDCG@5
0,tfidf,False,False,0.856,0.8378,0.8463
1,tfidf,False,True,0.880,0.8485,0.8689
2,tfidf,True,False,0.872,0.8521,0.8544
3,tfidf,True,True,0.904,0.8714,0.8842
4,bm25,False,False,0.864,0.8645,0.8772
5,bm25,False,True,0.872,0.8704,0.8907
6,bm25,True,False,0.936,0.9072,0.9242
7,bm25,True,True,0.944,0.9103,0.9370
